<a href="https://colab.research.google.com/github/har1eyk/96w_column_rearrangement/blob/master/rp3_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title Enter the construct sequences in FASTA format and hit `Runtime` -> `Run all`
batch_size = 4 # @param {"type":"integer"}
input_fasta = """
>CONSTR_000001
MTVFFVTRLVKKHDKLSKQQIEDFAEKLMTILFETYRSHWHSDCPSKGQAFRCIRINNNQ
NKDPILERACVESNVDFSHLGLPKEMTIWVDPFEVCCRYGEKNHPFTVASFKGRWEEWEL
YQQISYAVSRASSDVSSGTSCDEESCGSHHHHHH
>CONSTR_000002
MDYTKPLEHPPVKRNEEAQVHDKLNSGMVSNMEGTAGGERPSVVNGDSGKSGGVGDPREP
LGCLQEGSGCHPTTESFEKSVREDASPLPHVCCCKQDALILQRGLHHEDGSQHIGLLHPG
DRGPDHEYVLVEEAECGSHHHHHH
>CONSTR_000003
MHHHHHHENLYFQGSLEVRGQLQSALLILGEPKEGGMPMNISIMPSSLQMKTPEGCTEIQ
LPAEVRLVPSSCRGLQFVVGDGLHLRLQTQAKLGTKLISMFNQSSQTQE
>CONSTR_000004
MECPEGQLPISSENDSTPTVSTSEVTSQQEPQILVDRGSETTYESSADIAGDEGTQIPAD
EDTQTDADSSAQAAAQAPENFQEGKDMSESQDEVPDEVENGSHHHHHH
>CONSTR_000005
MSTAPSEDIWKKFELVPSPPTSPPWGLGPGAGDPAPGIGPPEPWPGGCTGDEAESRGHSK
GWGRNYASIIRRDCMWSGFSARERLERAVSDRLAPGAPRGNPPKASAAPDCTPSLEAGNP
APAAPCPLGEPKTQACSGSESPSDSENEEIDVVTVEKRQSLGIRKPVTITVRADPLDPCM
KHFHGSHHHHHH
>CONSTR_000006
MEKARHETFAAEMRQNDKIMCILENRKKRDRKNLCRAINDFQQSFQKPETRREFDLSDPL
ALKKDLPARQSDNDVRNTISGMQGSHHHHHH
>CONSTR_000007
MLMKKAYELSVLCDCEIALIIFNSANRLFQYASTDMDRVLLKYTEYSEPHESRTNTDILE
TLKRRGIGLDGPELEPDEGPEEPGEKFRRLAGEGGDPGSHHHHHH
>CONSTR_000008
MPTESASCSTARQTKQKRKSHSLSIRRTNSSEQERTGLPRDMLEGQDSKLPSSVRSTLLE
LFGQIEREFENLYIENLELRREIDTLNERLAAEGQAIDGAELSKGQLKTKASHSTSQLSQ
KLKTTYKASTSKIVSSFKTTTSRAACQLVKEYIGHRDGIWDVSVAKTQPVVLGTASADHT
ALLWSIETGKCLVKYAGHVGSVNSIKFHPSEQLALTASGDQTAHIWRYAVQLPTPQPVAD
TSISGEDEVECSDKDEPDLDGDVSSDCPTIRVPLTSLKSHQGVVIASDWLVGGKQAVTAS
WDRTANLYDVETSELVHSLTGHDQELTHCCTHPTQRLVVTSSRDTTFRLWDFRDPSIHSV
NVFQGHTDTVTSAVFTVGDNVVSGSDDRTVKVWDLKNMRSPIATIRTDSAINRINVCVGQ
KIIALPHDNRQVRLFDMSGVRLARLPRSSRQGHRRMVCCSAWSEDHPVCNLFTCGFDRQA
IGWNINIPALLQEKGSHHHHHH
>CONSTR_000009
MHHHHHHENLYFQGSPTESASCSTARQTKQKRKSHSLSIRRTNSSEQERTGLPRDMLEGQ
DSKLPSSVRSTLLELFGQIEREFENLYIENLELRREIDTLNERLAAEGQAIDGAELSKGQ
LKTKASHSTSQLSQKLKTTYKASTSKIVSSFKTTTSRAACQLVKEYIGHRDGIWDVSVAK
TQPVVLGTASADHTALLWSIETGKCLVKYAGHVGSVNSIKFHPSEQLALTASGDQTAHIW
RYAVQLPTPQPVADTSISGEDEVECSDKDEPDLDGDVSSDCPTIRVPLTSLKSHQGVVIA
SDWLVGGKQAVTASWDRTANLYDVETSELVHSLTGHDQELTHCCTHPTQRLVVTSSRDTT
FRLWDFRDPSIHSVNVFQGHTDTVTSAVFTVGDNVVSGSDDRTVKVWDLKNMRSPIATIR
TDSAINRINVCVGQKIIALPHDNRQVRLFDMSGVRLARLPRSSRQGHRRMVCCSAWSEDH
PVCNLFTCGFDRQAIGWNINIPALLQEK
>CONSTR_000010
MRDEIATTVFFVTRLVKKHDKLSKQQIEDFAEKLMTILFETYRSHWHSDCPSKGQAFRCI
RINNNQNKDPILERACVESNVDFSHLGLPKEMTIWVDPFEVCCRYGEKNHPFTVASFKGR
WEEWELYQQISYAVSRASSDVSSGTSCDEESCSKEPRVIPKVSNPKSIYQVENLKQPFQS
WLQIPRKKNVVDGRVGLLGNTYHGSQKHPKCYRPAMHRLDRILGSHHHHHH
"""

In [2]:
%%bash

set -e

pip install RP3Net 'torchvision==0.20.1' 'torchao>=0.16.0'
wget -nv -nc https://ftp.ebi.ac.uk/pub/software/RP3Net/v0.1/checkpoints/rp3net_v0.1_d.ckpt

In [3]:
#@title Imports (with compatibility fix)
import re
import io
import pandas as pd
import torch
import torch.utils._pytree

# Compatibility Fix 1: Manually add sub-byte attributes to torch if missing
for i in range(1, 8):
    attr = f'int{i}'
    if not hasattr(torch, attr):
        setattr(torch, attr, torch.int8)

# Compatibility Fix 2: Mock missing pytree registration attribute
if not hasattr(torch.utils._pytree, 'register_constant'):
    torch.utils._pytree.register_constant = lambda x: x

import torchao
from tqdm.notebook import tqdm

# Diagnostic check
print(f"Current torch version: {torch.__version__}")
print(f"Current torchao version: {torchao.__version__}")

import RP3Net as rp3
RE_FASTA_HEADER = re.compile(r'^>([\w\-.:#*]+)') # https://www.ncbi.nlm.nih.gov/genbank/fastaformat/

Current torch version: 2.5.1+cu124
Current torchao version: 0.17.0


In [4]:
#@title Helper functions
def iter_fasta(io):
    fasta_id, sequence = None, None
    for line in io:
        line = line.strip()
        if len(line) == 0:
            continue
        m = RE_FASTA_HEADER.match(line)
        if m:
            if fasta_id is not None:
                yield fasta_id, ''.join(sequence)
            sequence = []
            fasta_id = m.group(1)
        else:
            sequence.append(line)
    if fasta_id is not None:
        yield fasta_id, ''.join(sequence)

def parse_fasta(s):
    return {id: seq for id, seq in iter_fasta(io.StringIO(s))}

def batches():
    fasta_map = parse_fasta(input_fasta)
    fasta_keys = list(fasta_map.keys())
    r = tqdm(range(0, len(fasta_map), batch_size), desc='RP3Net Inference')
    for i in r:
        yield {k: fasta_map[k] for k in fasta_keys[i:i + batch_size]}

In [5]:
#@title Load the model
m = rp3.load_model(rp3.RP3_DEFAULT_CONFIG, 'rp3net_v0.1_d.ckpt')


In [6]:
#@title Run the prediction (Auto-detect GPU/CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

m = m.to(device=device)
scores_map = dict()
for b in batches():
    scores_map |= m.predict(b, device=device)

Using device: cpu


RP3Net Inference:   0%|          | 0/3 [00:00<?, ?it/s]

In [7]:
#@title Print and save the results
df = pd.DataFrame([[id, score] for (id, score) in scores_map.items()], columns=['id', 'score'])
print(df)
df.to_csv("rp3_scores.csv", index=False)

              id     score
0  CONSTR_000001  0.685278
1  CONSTR_000002  0.971392
2  CONSTR_000003  0.930146
3  CONSTR_000004  0.973004
4  CONSTR_000005  0.925137
5  CONSTR_000006  0.977901
6  CONSTR_000007  0.733684
7  CONSTR_000008  0.009328
8  CONSTR_000009  0.010420
9  CONSTR_000010  0.434350


In [ ]:
import json

# Path to your notebook file
# Note: In Colab, the current notebook is usually not directly accessible as a file
# unless you've saved it to the file system or are running this from a local copy.
# If you just want to clean a specific file you uploaded or saved:
input_file = 'rp3_inference.ipynb' # Change this to your filename
output_file = 'rp3_inference_cleaned.ipynb'

try:
    with open(input_file, 'r', encoding='utf-8') as f:
        nb = json.load(f)

    # Remove the problematic widgets metadata
    if 'widgets' in nb.get('metadata', {}):
        del nb['metadata']['widgets']
        print("Successfully removed 'widgets' from metadata.")
    else:
        print("No 'widgets' key found in metadata.")

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(nb, f, indent=1)

    print(f"Cleaned notebook saved as: {output_file}")
except FileNotFoundError:
    print(f"Error: {input_file} not found. Please ensure you have saved the notebook to the Colab files sidebar first.")